In [1]:
import numpy as np

# Read text

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f'Characters: {len(text)}')

Characters: 1115393


In [4]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [6]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


In [7]:
data = encode(text)
data = np.array(data, dtype=np.longfloat).reshape(-1, 1)

In [8]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
#print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

In [9]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+1:i+seq_len+1, :])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(X_train)
print(f'{X_train_seq.shape}, {y_train_seq.shape}')

(1003845, 8, 1), (1003845, 8, 1)


In [10]:
"""class Head(nn.Module):

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        super().__init__()
        self.normalize_factor = head_size**0.5
        self.key = nn.Linear(n_embed, head_size)
        self.query = nn.Linear(n_embed, head_size)
        self.value = nn.Linear(n_embed, head_size)
        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        w = q @ k.transpose(-2, -1) / self.normalize_factor
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # using mask makes it decoder block, in encoder every token can communicate
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)

        output = w @ v
        return output"""

"class Head(nn.Module):\n\n    def __init__(self, n_embed, head_size, block_size, dropout=0.1):\n        super().__init__()\n        self.normalize_factor = head_size**0.5\n        self.key = nn.Linear(n_embed, head_size)\n        self.query = nn.Linear(n_embed, head_size)\n        self.value = nn.Linear(n_embed, head_size)\n        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))\n\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        B, T, C = x.shape\n\n        k = self.key(x)\n        q = self.query(x)\n        v = self.value(x)\n\n        w = q @ k.transpose(-2, -1) / self.normalize_factor\n        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # using mask makes it decoder block, in encoder every token can communicate\n        w = F.softmax(w, dim=-1)\n        w = self.dropout(w)\n\n        output = w @ v\n        return output"

In [11]:
from dlfs.layers import DenseLayer
from dlfs.activation import Softmax

class SingleAttentionHead():

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        self.normalize_factor = head_size**0.5
        self.key = DenseLayer(n_embed, head_size)
        self.query = DenseLayer(n_embed, head_size)
        self.value = DenseLayer(n_embed, head_size)
        self.tril = np.tril(np.ones((block_size, block_size)))
        self.softmax = Softmax()
        #self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        print(f'x shape: {x.shape}')

        self.key.forward(x)
        self.query.forward(x)
        self.value.forward(x)

        self.k = self.key.output
        self.q = self.query.output
        self.v = self.value.output
        
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor
        mask_condition = self.tril[:T, :T] == 0
        self.w[:,mask_condition] = -np.inf
        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.output = np.matmul(self.w, self.v)

    def backward(self, delta):

        # Step 1: Gradient of the loss with respect to w (attention weights)
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        # Step 2: Gradient of the loss with respect to softmax input (logits)
        self.softmax.backward(d_w)  # Softmax backward pass
        d_w = self.softmax.dinputs

        # Step 3: Gradient of the loss with respect to w (before softmax)
        d_w = d_w * (self.w > 0).astype(float)  # Masking out invalid values from softmax

        # Step 4: Gradients w.r.t. key and query using the chain rule
        d_q = np.matmul(d_w, self.k)  # shape: (B, T, head_size)
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q)  # shape: (B, T, head_size)

        # Step 5: Update the key, query, and value parameters using the gradients
        # Gradient for the key (d_k) and query (d_q) go through the dense layers
        d_k_input = self.key.backward(d_k)
        d_q_input = self.query.backward(d_q)
        d_v_input = self.value.backward(np.matmul(d_w, self.v))

        d_k_input = self.key.dinputs
        d_q_input = self.query.dinputs
        d_v_input = self.value.dinputs

        # Step 6: Propagate the gradient further to the previous layer (if needed)

        return d_k_input, d_q_input, d_v_input  # Gradients w.r.t. the input to the dense layers

In [12]:
embedding = DenseLayer(1, 192)
head = SingleAttentionHead(n_embed=192, head_size=4, block_size=8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
#print(x.shape)
embedding.forward(x)
head.forward(embedding.output)
print(head.output.shape)

x shape: (1, 8, 192)
(1, 8, 4)


In [13]:
delta = np.random.rand(1, 8, 4)
a, b, c = head.backward(delta)

In [14]:
print(a.shape)
print(b.shape)
print(c.shape)

(1, 8, 192)
(1, 8, 192)
(1, 8, 192)
